In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

csv_file = '/content/drive/MyDrive/review_table/user-item_table.csv'
df = pd.read_csv(csv_file, encoding='utf-8-sig')

df.tail()

,user,name_product,total_score
5253,ZZ바라기,롬앤_블러퍼지틴트_언로즈,5
5254,Zzang쭈돌,3CE_드롭글로우젤_RAWAPPLE,5
5255,Zzang쭈돌,3CE_드롭글로우젤_RAWAPPLE,5
5256,Zzang쭈돌,페리페라_잉크더에어리벨벳_너지금무화과,5
5257,zzzzxc,키스미_아이라이트블러리픽싱틴트_릿치레드,5


In [ ]:
user_item_matrix = df.pivot_table(index='user', columns='name_product', values='total_score', fill_value=0)
#user=user_ID
print(user_item_matrix)

name_product    네이처리퍼블릭_허니멜팅립_고소한코코넛    네이처리퍼블릭_허니멜팅립_무드플럼핑  \
user                                                          
 olyoonzl                        0.0                    0.0   
00****                           0.0                    0.0   
00나를보면행운가득00                     0.0                    0.0   
0118c****                        0.0                    0.0   
012345678                        0.0                    0.0   
...                              ...                    ...   
히잠                               0.0                    0.0   
히치                               0.0                    0.0   
히히하하                             0.0                    0.0   
히히해                              0.0                    0.0   
히히혀                              0.0                    0.0   

name_product    네이처리퍼블릭_허니멜팅립_무화과한스푼    네이처리퍼블릭_허니멜팅립_블랙체리슈  \
user                                                          
 olyoonzl                        0.0                  

In [ ]:
user_means = user_item_matrix.mean(axis=1)  # 각 사용자의 평균 평점

# 사용자별 평균 정규화
mean_normalized_matrix = user_item_matrix.sub(user_means, axis=0)  # 각 사용자의 평균 평점을 뺌

print("사용자별 평균 정규화 결과:")
print(mean_normalized_matrix)

사용자별 평균 정규화 결과:
name_product    네이처리퍼블릭_허니멜팅립_고소한코코넛    네이처리퍼블릭_허니멜팅립_무드플럼핑  \
user                                                          
 olyoonzl                  -0.007812              -0.007812   
00****                     -0.009766              -0.009766   
00나를보면행운가득00               -0.097656              -0.097656   
0118c****                  -0.009766              -0.009766   
012345678                  -0.009766              -0.009766   
...                              ...                    ...   
히잠                         -0.007812              -0.007812   
히치                         -0.019531              -0.019531   
히히하하                       -0.009766              -0.009766   
히히해                        -0.009766              -0.009766   
히히혀                        -0.007812              -0.007812   

name_product    네이처리퍼블릭_허니멜팅립_무화과한스푼    네이처리퍼블릭_허니멜팅립_블랙체리슈  \
user                                                          
 olyoonzl                  -0.007812  

In [ ]:
r_min = mean_normalized_matrix.min().min()  # 전체 매트릭스 최소값
r_max = mean_normalized_matrix.max().max()  # 전체 매트릭스 최대값

min_max_normalized_matrix = (mean_normalized_matrix - r_min) / (r_max - r_min)

print("\nMin-Max 정규화 결과:")
print(min_max_normalized_matrix)


Min-Max 정규화 결과:
name_product    네이처리퍼블릭_허니멜팅립_고소한코코넛    네이처리퍼블릭_허니멜팅립_무드플럼핑  \
user                                                          
 olyoonzl                   0.047637               0.047637   
00****                      0.047265               0.047265   
00나를보면행운가득00                0.030517               0.030517   
0118c****                   0.047265               0.047265   
012345678                   0.047265               0.047265   
...                              ...                    ...   
히잠                          0.047637               0.047637   
히치                          0.045404               0.045404   
히히하하                        0.047265               0.047265   
히히해                         0.047265               0.047265   
히히혀                         0.047637               0.047637   

name_product    네이처리퍼블릭_허니멜팅립_무화과한스푼    네이처리퍼블릭_허니멜팅립_블랙체리슈  \
user                                                          
 olyoonzl                   0.047637 

In [9]:
import numpy as np

# 사용자-아이템 매트릭스
matrix = user_item_matrix.values  # 원본 매트릭스 (Pandas DataFrame → NumPy 배열)
num_users, num_items = matrix.shape

# Matrix Factorization 설정
k = 20  # 잠재 요인 수
alpha = 0.01  # 학습률
beta = 0.02  # 정규화 항 (오버피팅 방지)
iterations = 15000  # 반복 횟수

# P, Q 행렬 초기화 (임의의 값으로 시작)
P = np.random.rand(num_users, k)
Q = np.random.rand(num_items, k)

# 평점 있는 항목의 위치 확인
non_zero_indices = np.argwhere(matrix > 0)

# SGD 학습
for step in range(iterations):
    for user, item in non_zero_indices:
        rating = matrix[user, item]

        # 예측 평점
        prediction = np.dot(P[user, :], Q[item, :])
        error = rating - prediction  # 예측 오차

        # SGD 업데이트
        P[user, :] += alpha * (error * Q[item, :] - beta * P[user, :])
        Q[item, :] += alpha * (error * P[user, :] - beta * Q[item, :])

    # 손실 계산 (옵션)
    loss = sum(
        (matrix[user, item] - np.dot(P[user, :], Q[item, :])) ** 2
        for user, item in non_zero_indices
    )
    loss += beta * (np.sum(np.square(P)) + np.sum(np.square(Q)))  # 정규화 항
    if step % 500 == 0:  # 500번마다 손실 출력
        print(f"Iteration {step}, Loss: {loss}")

# 최종 예측 매트릭스
predicted_ratings = np.dot(P, Q.T)

# 다시 DataFrame으로 변환
predicted_ratings_df = pd.DataFrame(predicted_ratings, index=user_item_matrix.index, columns=user_item_matrix.columns)

print("\nMatrix Factorization 결과 예측 평점:")
print(predicted_ratings_df)


Iteration 0, Loss: 4331.803180946129
Iteration 500, Loss: 475.8286636324267
Iteration 1000, Loss: 450.9909363347395
Iteration 1500, Loss: 433.3866215737987
Iteration 2000, Loss: 420.329210932237
Iteration 2500, Loss: 410.4171896513702
Iteration 3000, Loss: 402.774347818309
Iteration 3500, Loss: 396.8107952254283


KeyboardInterrupt: 

In [ ]:
matrix = user_item_matrix.values
user_ids = user_item_matrix.index
item_ids = user_item_matrix.columns

k = 2  # 잠재 요인 수, 필요에 따라 조정
u, sigma, vt = svds(matrix, k=k)

# Sigma를 대각 행렬로 변환
sigma = np.diag(sigma)

# 예측 평점 행렬 계산
predicted_ratings = np.dot(np.dot(u, sigma), vt)

# 예측 결과를 다시 DataFrame으로 변환
predicted_ratings_df = pd.DataFrame(predicted_ratings, index=user_ids, columns=item_ids)

# 시그모이드 함수 적용
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

predicted_ratings_sigmoid = sigmoid(predicted_ratings)

# 예측 결과를 다시 DataFrame으로 변환
predicted_ratings_df = pd.DataFrame(predicted_ratings_sigmoid, index=user_ids, columns=item_ids)

print("\n시그모이드 함수를 적용한 SVD 예측 매트릭스:")
print(predicted_ratings_df)



시그모이드 함수를 적용한 SVD 예측 매트릭스:
name_product    네이처리퍼블릭_허니멜팅립_고소한코코넛    네이처리퍼블릭_허니멜팅립_무드플럼핑  \
user                                                          
 olyoonzl                   0.500001                    0.5   
00****                      0.500000                    0.5   
00나를보면행운가득00                0.500003                    0.5   
0118c****                   0.500000                    0.5   
012345678                   0.500000                    0.5   
...                              ...                    ...   
히잠                          0.500000                    0.5   
히치                          0.500000                    0.5   
히히하하                        0.500000                    0.5   
히히해                         0.500000                    0.5   
히히혀                         0.500000                    0.5   

name_product    네이처리퍼블릭_허니멜팅립_무화과한스푼    네이처리퍼블릭_허니멜팅립_블랙체리슈  \
user                                                          
 olyoonzl                 

In [ ]:
import pandas as pd
import numpy as np
from scipy.sparse.linalg import svds

R = user_item_matrix.values

k = 20
U, sigma, Vt = svds(R, k=k)

sigma = np.diag(sigma)

user_factors = np.dot(U, sigma)  # 사용자 벡터 행렬
item_factors = Vt.T              # 제품 벡터 행렬


print("User Factors (사용자 벡터):\n", user_factors)
print("Item Factors (제품 벡터):\n", item_factors)

User Factors (사용자 벡터):
 [[-4.18479036e-02  8.42603257e-02  2.08746727e-01 ... -6.33946231e-02
  -1.10487391e-03 -2.24960809e-02]
 [ 5.48399688e-02 -6.17153498e-02 -2.66865107e-01 ... -2.13268678e-01
  -7.81749937e-04 -2.39211191e-03]
 [ 2.21474236e-02 -2.59955555e-02 -4.12572182e-01 ... -1.46421786e+00
  -1.95242472e-03 -5.38753449e-02]
 ...
 [ 1.87278153e-04 -2.49767537e-05  1.05722838e-04 ... -1.56029853e-04
  -1.03457003e-04 -1.57954365e-07]
 [ 5.90052458e-03  4.71217867e-03  8.38046337e-03 ... -4.72874217e-03
  -1.23483836e-04 -1.17426002e-03]
 [ 3.23532889e-01 -4.16267520e-01  6.10764967e-01 ... -7.53552963e-02
  -2.84904933e-03 -5.12138147e-04]]
Item Factors (제품 벡터):
 [[-3.97946070e-03  2.09422347e-03  2.67482411e-03 ... -5.42623762e-03
  -1.00867063e-04 -2.46973032e-04]
 [ 2.77286831e-18 -5.78398875e-21 -2.82631319e-18 ...  1.82295831e-17
  -9.96591107e-18  5.75393964e-19]
 [ 1.09679938e-02 -1.23430700e-02 -5.33730215e-02 ... -4.26537355e-02
  -1.56349987e-04 -4.78422382e-04]
 .

In [ ]:
predicted_ratings = np.dot(user_factors, item_factors.T)

print("Predicted Ratings:\n", predicted_ratings)

Predicted Ratings:
 [[ 1.99445191e-03  3.82433109e-19 -1.46074086e-02 ... -6.17242304e-03
  -9.06391634e-03 -1.78807606e-18]
 [ 3.20892632e-01  4.38406667e-17  4.62171539e+00 ... -3.06850572e-03
   2.71322283e-02 -4.73164440e-17]
 [-1.39846986e-02 -7.29540966e-17  1.60130230e-01 ...  5.66554433e-03
  -6.03196442e-03  3.69010998e-17]
 ...
 [ 3.48203012e-06  1.39749316e-20 -6.99336474e-05 ... -4.39722859e-05
  -7.33747186e-05 -2.30169056e-20]
 [ 1.22983401e-04  6.57349176e-19 -2.29411125e-04 ...  1.20693930e-03
   1.83578293e-03 -1.37259194e-19]
 [-2.67783806e-03  1.70487810e-19 -2.76177826e-02 ... -1.76992700e-03
  -5.73206872e-03  1.20424076e-18]]


In [ ]:
from scipy.special import expit

#시그모이드 함수 적용
sigmoid_predictions = expit(predicted_ratings)
print("Sigmoid of Predicted Ratings:\n", sigmoid_predictions)

Sigmoid of Predicted Ratings:
 [[0.50049861 0.5        0.49634821 ... 0.4984569  0.49773404 0.5       ]
 [0.57954178 0.5        0.99025989 ... 0.49923287 0.50678264 0.5       ]
 [0.49650388 0.5        0.53994723 ... 0.50141638 0.49849201 0.5       ]
 ...
 [0.50000087 0.5        0.49998252 ... 0.49998901 0.49998166 0.5       ]
 [0.50003075 0.5        0.49994265 ... 0.50030173 0.50045895 0.5       ]
 [0.49933054 0.5        0.49309599 ... 0.49955752 0.49856699 0.5       ]]


**EX**


In [ ]:
import pandas as pd

csv_file = '/content/drive/MyDrive/review_table/EX1.csv'
df = pd.read_csv(csv_file, encoding='utf-8-sig')\

df.head()

,user_ID,name_product,total_score
0,1,3CE_블러워터틴트_DEARMARCH,5
1,2,3CE_블러워터틴트_LAYDOWN,5
2,2,딘토_블러글로이립틴트_피덴티아,5
3,3,딘토_블러글로이립틴트_피덴티아,4
4,4,딘토_블러글로이립틴트_피덴티아,5


In [ ]:
user_item_matrix = df.pivot_table(index='user_ID', columns='name_product', values='total_score', fill_value=0)

print(user_item_matrix)

name_product  3CE_블러워터틴트_DEARMARCH  3CE_블러워터틴트_LAYDOWN  딘토_블러글로이립틴트_피덴티아
user_ID                                                                 
1                              5.0                 0.0               0.0
2                              0.0                 5.0               5.0
3                              0.0                 0.0               4.0
4                              5.0                 2.0               5.0


In [ ]:
import pandas as pd
import numpy as np
from scipy.sparse.linalg import svds

R = user_item_matrix.values

k = 2
U, sigma, Vt = svds(R, k=k)

sigma = np.diag(sigma)

user_factors = np.dot(U, sigma)  # 사용자 벡터 행렬
item_factors = Vt.T              # 제품 벡터 행렬


print("User Factors (사용자 벡터):\n", user_factors)
print("Item Factors (제품 벡터):\n", item_factors)

User Factors (사용자 벡터):
 [[-4.38464265 -2.31742297]
 [ 3.39632514 -6.04287652]
 [ 1.30987903 -3.0796    ]
 [-2.04370332 -7.04427357]]
Item Factors (제품 벡터):
 [[-0.87692853 -0.46348459]
 [ 0.35179527 -0.4386753 ]
 [ 0.32746976 -0.7699    ]]


In [ ]:
# 내적
predicted_ratings = np.dot(user_factors, item_factors.T)

print("Predicted Ratings:\n", predicted_ratings)

Predicted Ratings:
 [[ 4.91910807 -0.52590032  0.34834608]
 [-0.17755424  3.84567181  5.7646044 ]
 [ 0.27867686  1.81175371  2.7999298 ]
 [ 5.05709402  2.37118369  4.75413519]]


In [ ]:
from scipy.special import expit

#시그모이드 함수 적용
sigmoid_predictions = expit(predicted_ratings)
print("Sigmoid of Predicted Ratings:\n", sigmoid_predictions)


Sigmoid of Predicted Ratings:
 [[0.99274734 0.37147358 0.58621645]
 [0.45572769 0.97907517 0.99687317]
 [0.56922181 0.85957369 0.94267203]
 [0.99367622 0.91460336 0.99145761]]
